# Dataset Cleaning pipeline — AI vs AI : Co Evolving AI Agents for Cyber Physical Defense of Smart Buildings


covering all 5 datasets: **CICIoT2023, CASAS, Ghost in the Building, TON_IoT, Bristol**.


**Always run the Setup cell first**, every session and after any crash/restart — Colab wipes variables and local files on disconnect, and this has been the single biggest source of repeated errors while building this notebook.

## Setup — run this first, every time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, random, zipfile, shutil, requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = '/content/drive/MyDrive/hisn_datasets'
RAW = f'{BASE}/raw'
CLEAN = f'{BASE}/cleaned'
os.makedirs(RAW, exist_ok=True)
os.makedirs(CLEAN, exist_ok=True)

def clean_dataset(df, name):
    """Generic first-pass cleaning: drop empty rows, dedupe, standardize column names."""
    before = len(df)
    df = df.dropna(how='all')
    df = df.drop_duplicates()
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    after = len(df)
    print(f'{name}: {before} -> {after} rows ({before - after} removed)')
    return df

def cap_home_rows(df, max_rows, group_col='home_id'):
    """Cap rows per group (e.g. per home) so no single one dominates the dataset."""
    return df.groupby(group_col, group_keys=False)[df.columns.tolist()].apply(
        lambda g: g.sample(n=min(len(g), max_rows), random_state=42)
    )

print('Setup complete. BASE =', BASE)

---
# 1. CICIoT2023

**Source:** Kaggle mirror of the UNB CIC IoT 2023 dataset (47M+ labeled network flows, 105 real IoT devices, 33 attack categories).
**Plan:** download to local disk (never straight to Drive — it's too large), filter to the 10 categories relevant to this project's attack types, clean, save only the small filtered result to Drive.

In [ ]:
# --- Kaggle credentials
!pip install -q --upgrade kaggle
os.environ['KAGGLE_USERNAME'] = 'deem45'
os.environ['KAGGLE_KEY'] = 'dvcmla'

LOCAL_CIC = '/content/ciciot2023_raw'
os.makedirs(LOCAL_CIC, exist_ok=True)

!kaggle datasets download -d madhavmalhotra/unb-cic-iot-dataset -p {LOCAL_CIC} --force

In [ ]:
# --- Verify the download is a valid
zip_path = f'{LOCAL_CIC}/unb-cic-iot-dataset.zip'
print('Size in MB:', os.path.getsize(zip_path) / (1024*1024))
print('Valid zip:', zipfile.is_zipfile(zip_path))

In [ ]:
# --- Extract and find every CSV file
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(LOCAL_CIC)

csv_path = f'{LOCAL_CIC}/wataiData/csv'
all_csv_files = glob.glob(f'{csv_path}/**/*.csv', recursive=True)
print('Total CSV part files found:', len(all_csv_files))

In [ ]:
# --- The 10 categories that map to this project's attack types (flood, spoofing, command-injection, benign)
KEEP_LABELS = [
    'DDoS-TCP_Flood', 'DDoS-SYN_Flood', 'DDoS-UDP_Flood',
    'DoS-TCP_Flood', 'DoS-SYN_Flood', 'DoS-UDP_Flood',
    'DNS_Spoofing', 'MITM-ArpSpoofing',
    'CommandInjection',
    'BenignTraffic',
]

# --- Clean + filter every part file, writing incrementally so nothing huge sits in memory at once
output_path = f'{LOCAL_CIC}/ciciot2023_filtered_clean.csv'
first_write = True
total_kept = 0
total_seen = 0

for i, file in enumerate(all_csv_files):
    df_part = pd.read_csv(file)
    total_seen += len(df_part)

    label_col = [c for c in df_part.columns if 'label' in c.lower()][0]
    df_part = df_part[df_part[label_col].isin(KEEP_LABELS)]

    if len(df_part) > 0:
        df_part_clean = clean_dataset(df_part, f'part {i+1}/{len(all_csv_files)}')
        df_part_clean.to_csv(output_path, mode='a', header=first_write, index=False)
        first_write = False
        total_kept += len(df_part_clean)

    if (i + 1) % 20 == 0:
        print(f'--- Progress: {i+1}/{len(all_csv_files)} files, {total_kept} rows kept so far ---')

print(f'\nDone. Total rows seen: {total_seen}, total rows kept: {total_kept}')

In [ ]:
# --- Verify WITHOUT loading the whole 6+ GB file into memory
with open(output_path) as f:
    row_count = sum(1 for _ in f) - 1
print('Row count:', row_count)

label_counts = {}
for chunk in pd.read_csv(output_path, chunksize=500_000):
    label_col = [c for c in chunk.columns if 'label' in c.lower()][0]
    for label, count in chunk[label_col].value_counts().items():
        label_counts[label] = label_counts.get(label, 0) + count
print(label_counts)

In [ ]:
# --- Copy the small filtered result to Drive (since raw dataset is too large we wont be importing it to the drive)
shutil.copy(output_path, f'{CLEAN}/ciciot2023_filtered_clean.csv')
print('Copied to Drive:', f'{CLEAN}/ciciot2023_filtered_clean.csv')

---
# 2. CASAS Smart Home Datasets

**Source:** Zenodo record 15708568 — 81 homes, 18 years of ambient PIR/door/temperature event logs.
**Plan:** query the Zenodo API for real filenames, download `labeled_data.zip`, parse the raw event-log format, filter directly into motion/door/temperature while parsing, clean, balance across homes, save.

In [ ]:
# --- Get the real file list from Zenodo's API
response = requests.get('https://zenodo.org/api/records/15708568')
data = response.json()
file_map = {f['key']: f['links']['self'] for f in data['files']}
for f in data['files']:
    print(f['key'], '-', round(f['size'] / (1024*1024), 1), 'MB')

In [ ]:
LOCAL_CASAS = '/content/casas_raw'
os.makedirs(LOCAL_CASAS, exist_ok=True)

def download_file(filename):
    url = file_map[filename]
    path = f'{LOCAL_CASAS}/{filename}'
    print(f'Downloading {filename}...')
    r = requests.get(url, stream=True)
    with open(path, 'wb') as out:
        for chunk in r.iter_content(chunk_size=8192):
            out.write(chunk)
    print(f'Done: {filename}, size MB:', os.path.getsize(path) / (1024*1024))
    return path

labeled_path = download_file('labeled_data.zip')

with zipfile.ZipFile(labeled_path, 'r') as z:
    z.extractall(f'{LOCAL_CASAS}/labeled_data')

In [ ]:
# --- Explicit allow-lists for each sensor category (CASAS has ~70 other sensor types we don't want)
motion_sensors_actual = ['Bedroom', 'LivingRoom', 'Kitchen',
                          'BedroomAArea', 'LivingRoomAArea', 'KitchenAArea',
                          'BedroomBArea', 'LivingRoomBArea', 'KitchenArea',
                          'LivingRoomArea', 'HallwayA', 'HallwayB', 'HallwayC']
door_sensors = ['OutsideDoor', 'MainDoor', 'BedroomADoor', 'LaundryRoomADoor', 'BedroomBDoor']
temp_sensors = ['KitchenATemperature', 'BathroomBTemperature', 'BathroomATemperature', 'KitchenTemperature']

def parse_casas_file_filtered(filepath, home_id):
    """Parses one home's raw event log, filtering to relevant sensors AS it reads —
    never builds the full unfiltered table, which is what caused RAM crashes earlier."""
    motion_rows, door_rows, temp_rows = [], [], []
    with open(filepath) as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) < 4:
                continue
            date, time, sensor, state = parts[0], parts[1], parts[2], parts[3]
            row = {'home_id': home_id, 'date': date, 'time': time, 'sensor': sensor, 'state': state}
            if sensor in motion_sensors_actual:
                motion_rows.append(row)
            elif sensor in door_sensors:
                door_rows.append(row)
            elif sensor in temp_sensors:
                temp_rows.append(row)
    return pd.DataFrame(motion_rows), pd.DataFrame(door_rows), pd.DataFrame(temp_rows)

In [ ]:
labeled_dir = f'{LOCAL_CASAS}/labeled_data/labeled'
sample_files = [f for f in os.listdir(labeled_dir) if f.endswith('.csv')]

motion_out = f'{LOCAL_CASAS}/motion_all.csv'
door_out = f'{LOCAL_CASAS}/door_all.csv'
temp_out = f'{LOCAL_CASAS}/temp_all.csv'
first_write = True

for f in sample_files:
    home_id = f.replace('.csv', '')
    m, d, t = parse_casas_file_filtered(f'{labeled_dir}/{f}', home_id)
    m.to_csv(motion_out, mode='a', header=first_write, index=False)
    d.to_csv(door_out, mode='a', header=first_write, index=False)
    t.to_csv(temp_out, mode='a', header=first_write, index=False)
    first_write = False
    print(f'{home_id}: motion={len(m)}, door={len(d)}, temp={len(t)}')

print('Done — all three files written incrementally.')

In [ ]:
# --- Reload. Note: temp_out lost its header row during writing in earlier runs — read it headerless to be safe.
column_names = ['home_id', 'date', 'time', 'sensor', 'state']

df_motion = pd.read_csv(motion_out, header=0, names=column_names)
df_door = pd.read_csv(door_out, header=0, names=column_names)
df_temp = pd.read_csv(temp_out, header=None, names=column_names)

print(df_motion.shape, df_door.shape, df_temp.shape)

In [ ]:
# --- Clean up state values: strip activity-label artifacts (e.g. "OFFEat") and stray numeric noise from other sensor types
df_motion['state'] = df_motion['state'].astype(str).str.extract(r'^(ON|OFF)', expand=False)
df_door['state'] = df_door['state'].astype(str).str.extract(r'^(OPEN|CLOSE|ON|OFF)', expand=False)
df_motion = df_motion.dropna(subset=['state'])
df_door = df_door.dropna(subset=['state'])

# Standardize door states to one vocabulary (some sensors log OPEN/CLOSE, others ON/OFF, for the same real-world event)
state_map = {'OPEN': 'ON', 'CLOSE': 'OFF', 'ON': 'ON', 'OFF': 'OFF'}
df_door['state'] = df_door['state'].map(state_map)

df_temp['state'] = pd.to_numeric(df_temp['state'], errors='coerce')
df_temp = df_temp.dropna(subset=['state'])

df_motion_clean = clean_dataset(df_motion, 'CASAS motion')
df_door_clean = clean_dataset(df_door, 'CASAS door')
df_temp_clean = clean_dataset(df_temp, 'CASAS temperature')

In [ ]:
# --- Balance across homes (mv001 alone had 2M+ raw motion rows vs. a median of ~87K)
df_motion_balanced = cap_home_rows(df_motion_clean, 150_000)
df_door_balanced = cap_home_rows(df_door_clean, 10_000)
df_temp_balanced = cap_home_rows(df_temp_clean, 1_500)

print('Motion:', len(df_motion_clean), '->', len(df_motion_balanced))
print('Door:', len(df_door_clean), '->', len(df_door_balanced))
print('Temp:', len(df_temp_clean), '->', len(df_temp_balanced))

In [ ]:
# --- Save both the unbalanced (_clean) and home-balanced (_balanced) versions ---
df_motion_clean.to_csv(f'{CLEAN}/casas_motion_clean.csv', index=False)
df_door_clean.to_csv(f'{CLEAN}/casas_door_clean.csv', index=False)
df_temp_clean.to_csv(f'{CLEAN}/casas_temperature_clean.csv', index=False)

df_motion_balanced.to_csv(f'{CLEAN}/casas_motion_balanced.csv', index=False)
df_door_balanced.to_csv(f'{CLEAN}/casas_door_balanced.csv', index=False)
df_temp_balanced.to_csv(f'{CLEAN}/casas_temperature_balanced.csv', index=False)

print('All six CASAS files saved to Drive.')

**Note:** only 27 of the 81 homes have temperature sensors — expected, not a bug (matches the sensor list). Motion/door are ~50/50 ON-OFF balanced after cleaning, confirmed by direct check.

---
# 3. Ghost in the Building

**Source:** BACS-forensics GitHub repo, companion data to the "Ghost in the Building" spoofing-attack paper. One `.ods` workbook, one sheet per room (8 rooms, ~86K rows each, one day of per-second logging).
**Plan:** clone, convert the whole workbook to `.xlsx` via LibreOffice, load each room sheet, combine, clean, and split out a suspicious CO2 anomaly window found in room B0115-108.

In [ ]:
LOCAL_GHOST = '/content/ghost_building_raw'
os.makedirs(LOCAL_GHOST, exist_ok=True)

!git clone -q https://github.com/BACS-forensics/The-ghost-in-the-building.git {LOCAL_GHOST}/repo

# Convert the WHOLE workbook (all sheets) to xlsx in one shot — do not use odfpy per-sheet, it crashes on this file
!apt-get install -y libreoffice-calc -qq > /dev/null 2>&1
!soffice --headless --convert-to xlsx --outdir {LOCAL_GHOST}/repo {LOCAL_GHOST}/repo/Datasets_raw_2022-05-12.ods

xlsx_path = f'{LOCAL_GHOST}/repo/Datasets_raw_2022-05-12.xlsx'
sheet_names = pd.ExcelFile(xlsx_path, engine='openpyxl').sheet_names
print('Sheets:', sheet_names)

In [ ]:
# --- Load each room, rename columns POSITIONALLY (names aren't fully consistent across rooms), tag with room, combine
room_sheets = ['B0115-107', 'B0115-108', 'B0115-113', 'B0115-115',
               'B0115-116', 'B0115-117', 'B0115-118', 'B0115-119']

all_rooms = []
for sheet in room_sheets:
    df_room = pd.read_excel(xlsx_path, engine='openpyxl', sheet_name=sheet)
    df_room.columns = ['timestamp', 'motion', 'co2_ppm', 'temperature']
    df_room['room'] = sheet
    all_rooms.append(df_room)
    print(f'{sheet}: {df_room.shape}')

df_ghost = pd.concat(all_rooms, ignore_index=True)
df_ghost['timestamp'] = pd.to_datetime(df_ghost['timestamp'])
df_ghost = df_ghost.sort_values(['room', 'timestamp']).reset_index(drop=True)
print('\nTotal combined shape:', df_ghost.shape)

In [ ]:
df_ghost_clean = clean_dataset(df_ghost, 'Ghost in the Building')
print(df_ghost_clean[['motion', 'co2_ppm', 'temperature']].describe())

In [ ]:
# --- Room B0115-108 (a room the source paper names as an experiment location) shows a CO2 spike
#     concentrated almost entirely in the 00:00-01:00 hour — split it out as a flagged window,
#     NOT confirmed as a real attack (the file has no labels), but worth keeping separate from the baseline.
df_ghost_baseline = df_ghost_clean[~((df_ghost_clean['room'] == 'B0115-108') &
                                       (df_ghost_clean['timestamp'].dt.hour == 0))].copy()
df_ghost_flagged = df_ghost_clean[(df_ghost_clean['room'] == 'B0115-108') &
                                    (df_ghost_clean['timestamp'].dt.hour == 0)].copy()

print('Baseline rows:', len(df_ghost_baseline))
print('Flagged (potential test window) rows:', len(df_ghost_flagged))

In [ ]:
df_ghost_baseline.to_csv(f'{CLEAN}/ghost_building_baseline.csv', index=False)
df_ghost_flagged.to_csv(f'{CLEAN}/ghost_building_flagged_window.csv', index=False)

# Back up the raw source files
RAW_BACKUP = f'{RAW}/ghost_building'
os.makedirs(RAW_BACKUP, exist_ok=True)
shutil.copytree(f'{LOCAL_GHOST}/repo', RAW_BACKUP, dirs_exist_ok=True)
print('Saved cleaned + raw backup to Drive.')

---
# 4. TON_IoT

**Source:** UNSW Canberra (request-form gated — download the raw CSVs manually and place them in `raw/ton_iot/` in Drive before running this section).
**Plan:** load each device file, replace junk placeholder values with real NaN, coerce numeric-looking columns, fill missing values, dedupe, tag each row with its source device, combine.

In [ ]:
ton_raw_folder = f'{RAW}/ton_iot'
ton_clean_folder = f'{CLEAN}/ton_iot'
os.makedirs(ton_raw_folder, exist_ok=True)
os.makedirs(ton_clean_folder, exist_ok=True)

junk_values = ["-", "?", "nan", "NaN", "NA", "null", "None", " "]

ton_files = glob.glob(ton_raw_folder + "/*.csv")
print("Files found:", len(ton_files))
print(ton_files)

In [ ]:
# --- Clean every TON_IoT device file, tag with device name, combine
all_clean_tables = []

for file_path in ton_files:
    file_name = os.path.basename(file_path).replace(".csv", "")
    print("Cleaning:", file_name)

    data = pd.read_csv(file_path)
    data = data.replace(junk_values, np.nan)

    for column in data.columns:
        try:
            data[column] = pd.to_numeric(data[column])
        except Exception:
            pass

    for column in data.columns:
        if pd.api.types.is_numeric_dtype(data[column]):
            data[column] = data[column].fillna(data[column].mean())
        else:
            data[column] = data[column].fillna("unknown")

    data = data.drop_duplicates()
    data["device"] = file_name
    data.to_csv(f'{ton_clean_folder}/{file_name}_clean.csv', index=False)
    all_clean_tables.append(data)

print("\nAll TON_IoT files cleaned.")

ton_all = pd.concat(all_clean_tables, ignore_index=True)
ton_all.to_csv(f'{ton_clean_folder}/ton_iot_ALL_clean.csv', index=False)
print("Combined file saved. Total rows:", len(ton_all))

if "label" in ton_all.columns:
    print("\nLabel counts (0 = normal, 1 = attack):")
    print(ton_all["label"].value_counts())

**Relevant device files for this project:** `IoT_Garage_Door.csv` (maps to servo lock), `IoT_Motion_Light.csv` (maps to PIR/LED), `IoT_Thermostat.csv` and `IoT_Weather.csv` (map to DHT22) — these give real labeled attack data for 3 of the 5 components, rather than relying on synthetic injections alone.

---
# 5. Bristol Smart Building Dataset

**Source:** University of Bristol data repository (scripted download often blocked — download manually and place file(s) in `raw/bristol/` in Drive before running this section).
**Plan:** load, coerce time/value columns to numeric, clip physically-impossible sensor values to NaN, fill, dedupe, label everything "normal" (no attacks in this dataset), sort by time.

In [ ]:
bristol_raw_folder = f'{RAW}/bristol'
bristol_clean_folder = f'{CLEAN}/bristol'
os.makedirs(bristol_raw_folder, exist_ok=True)
os.makedirs(bristol_clean_folder, exist_ok=True)

bristol_files = glob.glob(bristol_raw_folder + "/**/*.csv", recursive=True)
print("Bristol files found:", len(bristol_files))

bristol_tables = [pd.read_csv(f) for f in bristol_files]
bristol = pd.concat(bristol_tables, ignore_index=True)
print("Total rows:", len(bristol))
print("Columns:", bristol.columns.tolist())

In [ ]:
# --- Clean: junk -> NaN, coerce numeric, clip impossible values, fill, dedupe, label as baseline/normal
bristol = bristol.replace(junk_values, np.nan)
print("Sensor types:", bristol["Sensor"].unique())

bristol["Time"] = pd.to_numeric(bristol["Time"], errors="coerce")
bristol["Value"] = pd.to_numeric(bristol["Value"], errors="coerce")
bristol["timestamp"] = pd.to_datetime(bristol["Time"], unit="s", errors="coerce")

# Physically impossible values -> NaN, not statistical outliers (Bristol has no attacks to preserve)
bristol.loc[(bristol["Sensor"] == "Temperature") & (bristol["Value"] > 70), "Value"] = np.nan
bristol.loc[(bristol["Sensor"] == "Temperature") & (bristol["Value"] < -20), "Value"] = np.nan
bristol.loc[(bristol["Sensor"] == "Humidity") & (bristol["Value"] > 100), "Value"] = np.nan
bristol.loc[(bristol["Sensor"] == "Humidity") & (bristol["Value"] < 0), "Value"] = np.nan

bristol["Value"] = bristol["Value"].fillna(bristol["Value"].mean())
bristol["DeviceId"] = bristol["DeviceId"].fillna("unknown")
bristol["Sensor"] = bristol["Sensor"].fillna("unknown")

print("Rows before dedup:", len(bristol))
bristol = bristol.drop_duplicates()
print("Rows after dedup:", len(bristol))

bristol["label"] = 0
bristol["type"] = "normal"
bristol = bristol.sort_values("timestamp").reset_index(drop=True)

bristol.to_csv(f'{bristol_clean_folder}/bristol_ALL_clean.csv', index=False)
print("\nSaved. Total rows:", len(bristol))
print("Missing values left:", bristol.isnull().sum().sum())

---
## Per-dataset notes

**CICIoT2023.** 47M+ labeled network flows, 105 real IoT devices, 33 attack categories from CIC/UNB. Filtered down to 10 categories (6 flood types, 2 spoofing types, command-injection, benign) matching this project's attack scope. Severe class imbalance — flood dominates, command-injection is thin (~5.4K rows) and should be supplemented with synthetic examples.

**CASAS.** 81 homes, 18 years of ambient PIR/door/temperature event logs from WSU. Filtered via explicit sensor allow-lists (raw data has ~70 unrelated sensor types — pressure pads, appliance sensors, etc.). Door states standardized from a mixed ON/OFF + OPEN/CLOSE vocabulary. Row counts capped per home to prevent a few large homes from dominating. Only 27/81 homes have temperature sensors.

**Ghost in the Building.** One day, 8 rooms, per-second PIR/CO2/temperature logging, companion data to a real spoofing-attack paper. Room B0115-108 shows a CO2 anomaly concentrated in a single hour, split out as a separately-flagged (not confirmed) window.

**TON_IoT.** Labeled IoT dataset from UNSW Canberra across several device types (fridge, garage door, GPS tracker, Modbus, motion light, thermostat, weather). The dataset has a class imbalance problem: 69% of rows are attacks and only 31% are normal — the opposite of what real-world building activity would show, indicating the dataset was deliberately constructed for attack-detection research rather than sampled to reflect natural frequency. Within attack types, backdoor, password, and injection each have 30,000+ rows, while XSS and scanning have under 6,000. The Garage Door device — the closest match to this project's servo lock — is especially imbalanced (18,777 attack vs. only 589 normal rows). Devices have different sensor columns, so the data is used per-device rather than as one combined table. These imbalances need to be addressed before training the defender (e.g., undersampling attacks or oversampling normal rows), particularly for the Garage Door subset given its relevance to the lock component.

**Bristol.** University of Bristol smart-building dataset, 8 IoT devices logging temperature, humidity, and other sensors roughly every 10 seconds over six months. All data is normal — no attacks — as expected, used as a clean baseline. Values fall within realistic indoor ranges (temperature 20–31°C, humidity 41–68%). A representative sample of the full six-month dataset was cleaned; no issues found beyond removing a small number of physically impossible values and duplicate rows.